# Cardiovascular Disease Risk Prediction from NHANES Data

This notebook develops and evaluates a machine learning pipeline that estimates the probability of cardiovascular disease (CVD) from dietary intake, anthropometric measurements, blood pressure, blood biomarkers, smoking status, diabetes status, kidney function, and demographics, using data from the National Health and Nutrition Examination Survey (NHANES).

**Approach inspired by:** Ahiduzzaman & Hasan (2025), *"Interpretable machine learning for cardiovascular risk prediction,"* PLoS One. This is an independent implementation with several methodological additions described throughout, including leakage-safe cross-validation, hyperparameter tuning, probability calibration, and a subgroup fairness audit. Smoking status, HDL cholesterol, diabetes status, and estimated kidney function (eGFR) are included here as additional predictors beyond the original paper's scope, reflecting their established roles in cardiovascular risk assessment (e.g., in the ACC/AHA Pooled Cohort Equations and AHA PREVENT risk calculators).

## Environment Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")  # suppress non-critical library warnings

import os
import pickle
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

# Missing-data imputation
from sklearn.experimental import enable_iterative_imputer  # must be imported first
from sklearn.impute import IterativeImputer

# Feature selection and model tuning
from sklearn.feature_selection import RFE
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler

# Candidate classifiers
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# Class-imbalance correction
from imblearn.over_sampling import RandomOverSampler
from imblearn.pipeline import Pipeline as ImbPipeline

# Evaluation metrics
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, brier_score_loss
)
from sklearn.calibration import calibration_curve, CalibratedClassifierCV

# Model interpretability
import shap
import lime
import lime.lime_tabular

RANDOM_SEED = 42  # fixed seed for reproducibility
os.makedirs("figures", exist_ok=True)

## 1. Data Loading

In [ ]:
FILE_PATH = "data/nhanes_cvd_extract.xlsx"

raw_data = pd.read_excel(FILE_PATH, sheet_name="Sheet1")
print(f"Loaded {raw_data.shape[0]:,} people and {raw_data.shape[1]} columns of information.")

## 2. Outcome Definition

The target variable is a binary indicator of cardiovascular disease (CVD) status. A participant is coded 1 if they reported a physician diagnosis of angina, coronary heart disease, congestive heart failure, heart attack, or stroke, and 0 if they reported none of these. This composite outcome (`cvd`) is precomputed in the source extract.

In [ ]:
print("CVD outcome distribution before cleaning:")
print(raw_data["cvd"].value_counts(dropna=False))

## 3. Cohort Definition

The analysis is restricted to adults (age 18+), consistent with the cardiovascular risk literature this project draws on.

In [ ]:
adults = raw_data[raw_data["RIDAGEYR"] >= 18].copy()
print(f"Adults remaining: {adults.shape[0]:,} out of {raw_data.shape[0]:,} total people.")

## 4. Candidate Predictors

Forty-two candidate predictors are considered, spanning five domains: dietary intake (macronutrients, vitamins, minerals), anthropometric/clinical measurements (BMI, waist circumference, blood pressure, total and HDL cholesterol, CRP, estimated kidney function), demographics (age), and two established cardiovascular risk factors not present in the original paper -- smoking status and diabetes status.

- **Smoking status** is derived from NHANES questions `SMQ020` and `SMQ040`, encoded as 0 (never smoked), 1 (former smoker), or 2 (current smoker).
- **Diabetes status** is derived from `DIQ010` ("has a doctor told you that you have diabetes"), encoded as 0 (no) or 1 (yes).
- **Estimated glomerular filtration rate (eGFR)**, a measure of kidney function, is calculated from serum creatinine, age, and sex using the 2021 CKD-EPI creatinine equation -- the current race-free standard recommended by the National Kidney Foundation and American Society of Nephrology, replacing an earlier version of the equation that used a race coefficient.

These four additions mirror inputs used by established clinical risk tools such as the ACC/AHA Pooled Cohort Equations and the AHA PREVENT calculator.

Two additional variables -- sex and race/ethnicity -- are retained but explicitly excluded from the predictor set. They are used exclusively in the post-hoc fairness audit (Section 13) to evaluate whether model performance is consistent across demographic subgroups. Notably, the PREVENT equations similarly removed race as a direct model input in 2023, for related fairness considerations.

In [ ]:
DIETARY_VARS = [
    "DR1TPROT", "DR1TCARB", "DR1TSUGR", "DR1TFIBE", "DR1TSFAT", "DR1TMFAT",
    "DR1TPFAT", "DR1TCHOL", "DR1TBCAR", "DR1TCRYP", "DR1TLYCO", "DR1TVB1",
    "DR1TVB2", "DR1TNIAC", "DR1TVB6", "DR1TFOLA", "DR1TFF", "DR1TIRON",
    "DR1TCHL", "DR1TVB12", "DR1TVC", "DR1TVD", "DR1TATOC", "DR1TVK",
    "DR1TCALC", "DR1TPHOS", "DR1TMAGN", "DR1TZINC", "DR1TCOPP", "DR1TSODI",
    "DR1TPOTA", "DR1TSELE", "DR1TMOIS",
]
CLINICAL_VARS = ["BMXBMI", "BMXWAIST", "SBP1", "DBP1", "LBXTC", "HDL", "LBXHSCRP", "eGFR"]
DEMOGRAPHIC_VARS = ["RIDAGEYR"]
BEHAVIORAL_VARS = ["smoking_status", "diabetes_status"]
ALL_PREDICTORS = DIETARY_VARS + CLINICAL_VARS + DEMOGRAPHIC_VARS + BEHAVIORAL_VARS

# These two are NEVER used as model predictors, but used afterward purely to
# audit the model for fairness across subgroups.
FAIRNESS_CHECK_VARS = ["RIAGENDR", "RIDRETH3"]

READABLE_NAMES = {
    "DR1TPROT": "Protein (g)", "DR1TCARB": "Carbohydrates (g)", "DR1TSUGR": "Total Sugars (g)",
    "DR1TFIBE": "Fiber (g)", "DR1TSFAT": "Saturated Fat (g)", "DR1TMFAT": "Monounsaturated Fat (g)",
    "DR1TPFAT": "Polyunsaturated Fat (g)", "DR1TCHOL": "Dietary Cholesterol (mg)",
    "DR1TBCAR": "Beta-Carotene (mcg)", "DR1TCRYP": "Beta-Cryptoxanthin (mcg)", "DR1TLYCO": "Lycopene (mcg)",
    "DR1TVB1": "Thiamin / Vitamin B1 (mg)", "DR1TVB2": "Riboflavin / Vitamin B2 (mg)",
    "DR1TNIAC": "Niacin / Vitamin B3 (mg)", "DR1TVB6": "Vitamin B6 (mg)",
    "DR1TFOLA": "Total Folate (mcg)", "DR1TFF": "Food Folate (mcg)", "DR1TIRON": "Iron (mg)",
    "DR1TCHL": "Total Choline (mg)", "DR1TVB12": "Vitamin B12 (mcg)", "DR1TVC": "Vitamin C (mg)",
    "DR1TVD": "Vitamin D (mcg)", "DR1TATOC": "Vitamin E (mg)", "DR1TVK": "Vitamin K (mcg)",
    "DR1TCALC": "Calcium (mg)", "DR1TPHOS": "Phosphorus (mg)", "DR1TMAGN": "Magnesium (mg)",
    "DR1TZINC": "Zinc (mg)", "DR1TCOPP": "Copper (mg)", "DR1TSODI": "Sodium (mg)",
    "DR1TPOTA": "Potassium (mg)", "DR1TSELE": "Selenium (mcg)", "DR1TMOIS": "Food Moisture / Water Content (g)",
    "BMXBMI": "Body Mass Index (BMI)", "BMXWAIST": "Waist Circumference (cm)",
    "SBP1": "Systolic Blood Pressure (mmHg)", "DBP1": "Diastolic Blood Pressure (mmHg)",
    "LBXTC": "Total Cholesterol (mg/dL)", "HDL": "HDL Cholesterol (mg/dL)",
    "LBXHSCRP": "C-Reactive Protein (CRP, mg/L)", "eGFR": "Estimated Kidney Function (eGFR)",
    "RIDAGEYR": "Age (years)",
    "smoking_status": "Smoking Status",
    "diabetes_status": "Diabetes Status",
}


def to_readable(columns):
    """Translate NHANES variable codes into plain-English labels."""
    return [READABLE_NAMES.get(c, c) for c in columns]


print(f"Total candidate predictors: {len(ALL_PREDICTORS)}")

## 5. Missing Data Handling

Missing data are addressed in two stages:

1. **Complete-case filtering on dietary data, smoking status, diabetes status, and outcome status.** Participants missing an entire dietary recall interview, missing either behavioral questionnaire, or with unknown CVD status, are excluded rather than imputed. Both behavioral variables are categorical with minimal missingness among adults in this dataset (well under 5%), consistent with the treatment of smoking status.
2. **Multivariate imputation for clinical measurements.** For the remaining fraction of participants missing individual clinical values (e.g., a blood pressure reading, or HDL cholesterol and eGFR, both of which depend on a blood draw and have somewhat higher missingness than interview-based variables), values are imputed using `IterativeImputer`, which estimates each missing value from a participant's other observed variables rather than using univariate mean imputation.

In [ ]:
before_n = len(adults)
complete_case_data = adults.dropna(subset=DIETARY_VARS + BEHAVIORAL_VARS + ["cvd"]).copy()
print(f"Removed {before_n - len(complete_case_data):,} people missing dietary data, "
      f"smoking or diabetes status, or CVD status.")
print(f"Remaining: {len(complete_case_data):,} people. This is the final study cohort.")

print("\nFinal CVD outcome distribution:")
print((complete_case_data["cvd"].value_counts(normalize=True).round(3) * 100), "(percent)")

In [ ]:
missing_before = complete_case_data[CLINICAL_VARS].isna().mean() * 100
print("Percent missing in each clinical measurement, before imputation:")
print(missing_before.round(2))

imputer = IterativeImputer(max_iter=10, random_state=RANDOM_SEED)
complete_case_data[ALL_PREDICTORS] = imputer.fit_transform(complete_case_data[ALL_PREDICTORS])

print("\nMissing values remaining (should be 0):", complete_case_data[ALL_PREDICTORS].isna().sum().sum())

keep_cols = ALL_PREDICTORS + ["cvd"] + FAIRNESS_CHECK_VARS
df = complete_case_data[keep_cols].copy()
df["cvd"] = df["cvd"].astype(int)

## 6. Train/Test Split

The data are split 80/20 into training and test sets, stratified by outcome.

This split is performed **before** feature selection (Section 7). Performing feature selection on the full dataset prior to splitting would allow information from the test set to influence which variables are selected -- a data leakage issue that produces optimistic bias in the resulting performance estimates. The test set defined here is not used again until final model evaluation.

In [ ]:
X_all = df[ALL_PREDICTORS]
y_all = df["cvd"]

X_train_full, X_test_full, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.20, random_state=RANDOM_SEED, stratify=y_all
)

print(f"Training set: {len(X_train_full):,} observations")
print(f"Test set (held out until final evaluation): {len(X_test_full):,} observations")

## 7. Feature Selection

Recursive Feature Elimination (RFE) with a Random Forest base estimator is used to select the 20 most informative predictors from the 42 candidates, using **training data only**, consistent with the leakage-prevention rationale in Section 6.

In [ ]:
N_FEATURES_TO_KEEP = 20

rfe_selector = RFE(
    estimator=RandomForestClassifier(n_estimators=100, max_depth=10, random_state=RANDOM_SEED, n_jobs=1),
    n_features_to_select=N_FEATURES_TO_KEEP,
    step=2,
)
rfe_selector.fit(X_train_full, y_train)

selected_features = X_train_full.columns[rfe_selector.support_].tolist()
X_train = X_train_full[selected_features]
X_test = X_test_full[selected_features]

print(f"Selected {len(selected_features)} of {len(ALL_PREDICTORS)} candidate predictors:")
for f in selected_features:
    print(f"  - {READABLE_NAMES.get(f, f)}")

## 8. Class Imbalance Correction

The training data exhibit substantial class imbalance (~88% no-CVD / ~12% CVD). Without correction, models tend to default to predicting the majority class, achieving high accuracy while failing to identify true positive cases. This is addressed via random oversampling of the minority class.

**Implementation detail -- avoiding oversampling-induced leakage:** oversampling by duplication means resampled data can contain exact duplicate rows. If oversampling is applied once to the full training set, and cross-validation is subsequently performed on that oversampled data (as in hyperparameter search), a duplicate of a given observation can appear in both the training and validation partition of a fold, artificially inflating validation performance. This was confirmed empirically during development, where it produced an implausible cross-validated AUROC of 0.99.

The correction is to encapsulate oversampling as a pipeline step (via `imblearn.pipeline.Pipeline`) rather than as a one-time preprocessing operation, so that oversampling -- and, where applicable, feature scaling -- is refit independently within each cross-validation fold.

In [ ]:
def build_pipeline(base_model, needs_scaling=False):
    """Construct a pipeline combining oversampling, optional feature
    scaling, and the classifier as sequential steps. Encapsulating these
    as pipeline steps (rather than applying them once upfront) ensures
    they are refit independently within each cross-validation fold,
    avoiding the leakage issue described above."""
    steps = [("balance", RandomOverSampler(random_state=RANDOM_SEED))]
    if needs_scaling:
        steps.append(("rescale", StandardScaler()))
    steps.append(("predict", base_model))
    return ImbPipeline(steps)

print("Pipeline factory ready. Oversampling will be refit fresh within each fold.")

## 9. Model Training and Hyperparameter Tuning

Five classifiers are trained and compared: Logistic Regression, Random Forest, Support Vector Machine (RBF kernel), XGBoost, and LightGBM.

Hyperparameters for each model are selected via `RandomizedSearchCV` with cross-validation on the training set, rather than using fixed literature-reported defaults. Search space sizes and cross-validation fold counts are kept modest to maintain reasonable runtime on limited compute; for SVM specifically, the search is conducted on a random subsample of the training data (a standard practice given the poor scaling of RBF-kernel methods with sample size), with the selected configuration then refit on the full training set.

In [ ]:
PARAM_GRIDS = {
    "Logistic Regression": {"predict__C": [0.01, 0.1, 1, 10, 100], "predict__max_iter": [3000]},
    "Random Forest": {
        "predict__n_estimators": [100, 150, 250],
        "predict__max_depth": [6, 10, 14],
        "predict__min_samples_leaf": [1, 3, 5],
    },
    "SVM": {"predict__C": [0.1, 1, 5], "predict__gamma": ["scale", "auto"]},
    "XGBoost": {
        "predict__n_estimators": [150, 250, 350],
        "predict__max_depth": [4, 6, 8],
        "predict__learning_rate": [0.03, 0.05, 0.1],
    },
    "LightGBM": {
        "predict__n_estimators": [150, 250, 350],
        "predict__num_leaves": [15, 31, 63],
        "predict__learning_rate": [0.03, 0.05, 0.1],
    },
}

MODEL_BUILDERS = {
    "Logistic Regression": lambda: build_pipeline(LogisticRegression(), needs_scaling=True),
    "Random Forest": lambda: build_pipeline(RandomForestClassifier(random_state=RANDOM_SEED, n_jobs=1)),
    "SVM": lambda: build_pipeline(SVC(probability=False, random_state=RANDOM_SEED, cache_size=1000), needs_scaling=True),
    "XGBoost": lambda: build_pipeline(XGBClassifier(eval_metric="logloss", random_state=RANDOM_SEED, n_jobs=1)),
    "LightGBM": lambda: build_pipeline(LGBMClassifier(random_state=RANDOM_SEED, verbose=-1, n_jobs=1)),
}

N_SEARCH_ITERATIONS = {"Logistic Regression": 5, "Random Forest": 6, "SVM": 5, "XGBoost": 8, "LightGBM": 8}
SEARCH_SUBSAMPLE = {"SVM": 1500}

trained_models = {}
cv = StratifiedKFold(n_splits=2, shuffle=True, random_state=RANDOM_SEED)

for model_name, build_fn in MODEL_BUILDERS.items():
    print(f"Tuning {model_name} ...")

    if model_name in SEARCH_SUBSAMPLE:
        n_sub = min(SEARCH_SUBSAMPLE[model_name], len(X_train))
        sub_idx = X_train.sample(n=n_sub, random_state=RANDOM_SEED).index
        X_search, y_search = X_train.loc[sub_idx], y_train.loc[sub_idx]
    else:
        X_search, y_search = X_train, y_train

    search = RandomizedSearchCV(
        build_fn(), PARAM_GRIDS[model_name],
        n_iter=N_SEARCH_ITERATIONS[model_name], scoring="roc_auc",
        cv=cv, random_state=RANDOM_SEED, n_jobs=-1,
    )
    search.fit(X_search, y_search)

    final_model = build_fn()
    final_model.set_params(**search.best_params_)
    final_model.fit(X_train, y_train)

    trained_models[model_name] = {"pipeline": final_model, "best_params": search.best_params_}
    print(f"  Done. Selected hyperparameters: {search.best_params_}")

print("\nAll 5 models trained.")

## 10. Model Evaluation

Each model is evaluated once, on the held-out test set defined in Section 6. Six metrics are reported:

- **Accuracy** -- overall proportion of correct predictions (can be misleading under class imbalance).
- **Precision** -- proportion of predicted-positive cases that are true positives.
- **Recall (Sensitivity)** -- proportion of actual positive cases correctly identified.
- **Specificity** -- proportion of actual negative cases correctly identified.
- **F1-score** -- harmonic mean of precision and recall.
- **AUROC** -- probability that the model assigns a higher risk score to a randomly selected positive case than a randomly selected negative case; independent of any classification threshold, and the primary metric used for model comparison below.

In [ ]:
MODEL_ORDER = ["XGBoost", "Random Forest", "Logistic Regression", "LightGBM", "SVM"]

performance_results = {}
roc_curve_data = {}
risk_scores_by_model = {}

for model_name, info in trained_models.items():
    pipeline = info["pipeline"]
    predictions = pipeline.predict(X_test)
    risk_scores = (pipeline.predict_proba(X_test)[:, 1]
                   if hasattr(pipeline, "predict_proba")
                   else pipeline.decision_function(X_test))
    risk_scores_by_model[model_name] = risk_scores

    tn, fp, fn, tp = confusion_matrix(y_test, predictions).ravel()
    performance_results[model_name] = {
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions),
        "Recall (Sensitivity)": recall_score(y_test, predictions),
        "Specificity": tn / (tn + fp),
        "F1-score": f1_score(y_test, predictions),
        "AUROC": roc_auc_score(y_test, risk_scores),
    }
    fpr, tpr, _ = roc_curve(y_test, risk_scores)
    roc_curve_data[model_name] = (fpr, tpr, performance_results[model_name]["AUROC"])

performance_table = pd.DataFrame(performance_results).round(4)[MODEL_ORDER]
print("===================== MODEL SCORECARD =====================")
print(performance_table)

best_model_name = performance_table.loc["AUROC"].idxmax()
print(f"\nBest-performing model by AUROC: {best_model_name}")

In [ ]:
# Visual comparison of ROC curves across all 5 models.
plt.figure(figsize=(7, 6))
for model_name, (fpr, tpr, auc_value) in roc_curve_data.items():
    plt.plot(fpr, tpr, label=f"{model_name} (AUROC = {auc_value:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random guessing")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison Across Models")
plt.legend(loc="lower right", fontsize=9)
plt.tight_layout()
plt.savefig("figures/roc_curve_comparison.png", dpi=150)
plt.show()

## 11. Confidence Intervals via Bootstrap Resampling

A point estimate of AUROC does not indicate how much that estimate might vary under resampling. Bootstrap resampling of the test set (with replacement, 1,000 iterations) is used to construct 95% confidence intervals for each model's AUROC. Substantially overlapping intervals across models indicate that their performance is not statistically distinguishable on this test set.

In [ ]:
def bootstrap_auroc_ci(risk_scores_by_model, y_test, n_bootstrap=1000):
    y_arr = np.asarray(y_test)
    rng = np.random.RandomState(RANDOM_SEED)
    n = len(y_arr)
    results = {}
    for model_name, scores in risk_scores_by_model.items():
        scores_arr = np.asarray(scores)
        boot_scores = []
        for _ in range(n_bootstrap):
            idx = rng.randint(0, n, n)
            if len(np.unique(y_arr[idx])) < 2:
                continue
            boot_scores.append(roc_auc_score(y_arr[idx], scores_arr[idx]))
        results[model_name] = {
            "Median AUROC": np.median(boot_scores),
            "95% CI Lower": np.percentile(boot_scores, 2.5),
            "95% CI Upper": np.percentile(boot_scores, 97.5),
        }
    return pd.DataFrame(results).T.round(4).loc[[m for m in MODEL_ORDER if m in results]]

ci_table = bootstrap_auroc_ci(risk_scores_by_model, y_test)
print(ci_table)

## 12. Probability Calibration

AUROC measures discrimination (rank-ordering) but not calibration -- whether a predicted probability corresponds to an empirical frequency. A model can have strong discrimination while producing systematically biased probability estimates. Calibration is assessed by binning predictions and comparing mean predicted probability to observed event rate within each bin.

In [ ]:
plt.figure(figsize=(7, 7))
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Perfectly calibrated")

brier_scores_before = {}
for model_name, info in trained_models.items():
    pipeline = info["pipeline"]
    if not hasattr(pipeline, "predict_proba"):
        continue
    probs = pipeline.predict_proba(X_test)[:, 1]
    frac_pos, mean_pred = calibration_curve(y_test, probs, n_bins=10, strategy="quantile")
    plt.plot(mean_pred, frac_pos, marker="o", label=model_name)
    brier_scores_before[model_name] = brier_score_loss(y_test, probs)

plt.xlabel("Mean Predicted Probability")
plt.ylabel("Observed Event Rate")
plt.title("Calibration Curves (Before Correction)")
plt.legend(loc="upper left", fontsize=9)
plt.tight_layout()
plt.savefig("figures/calibration_before.png", dpi=150)
plt.show()

print("Brier scores before calibration (lower is better):")
print(pd.Series(brier_scores_before).round(4))

Predicted probabilities fall consistently below the diagonal reference line, indicating that every model **overestimates risk**. This is an expected consequence of training on oversampled data, where the effective prevalence of the positive class during training (~50%) is far higher than its true prevalence (~12%).

This is corrected via post-hoc calibration (Platt scaling / `CalibratedClassifierCV`), which recalibrates predicted probabilities against observed outcomes without altering the model's rank-ordering -- AUROC is expected to remain effectively unchanged.

In [ ]:
best_pipeline = trained_models[best_model_name]["pipeline"]
calibrated_model = None

if hasattr(best_pipeline, "predict_proba"):
    calibrated_model = CalibratedClassifierCV(best_pipeline, method="sigmoid", cv=3)
    calibrated_model.fit(X_train, y_train)

    probs_before = best_pipeline.predict_proba(X_test)[:, 1]
    probs_after = calibrated_model.predict_proba(X_test)[:, 1]

    print(f"{best_model_name} Brier score -- before: {brier_score_loss(y_test, probs_before):.4f}, "
          f"after: {brier_score_loss(y_test, probs_after):.4f}")
    print(f"{best_model_name} AUROC -- before: {roc_auc_score(y_test, probs_before):.4f}, "
          f"after: {roc_auc_score(y_test, probs_after):.4f}")

    plt.figure(figsize=(7, 7))
    plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Perfectly calibrated")
    frac_pos, mean_pred = calibration_curve(y_test, probs_after, n_bins=10, strategy="quantile")
    plt.plot(mean_pred, frac_pos, marker="o", color="tab:blue", label=f"{best_model_name} (calibrated)")
    plt.xlabel("Mean Predicted Probability")
    plt.ylabel("Observed Event Rate")
    plt.title("Calibration Curve (After Correction)")
    plt.legend(loc="upper left", fontsize=9)
    plt.tight_layout()
    plt.savefig("figures/calibration_after.png", dpi=150)
    plt.show()

## 13. Subgroup Performance Audit (Fairness)

**Objective:** evaluate whether model discrimination (AUROC) is consistent across demographic subgroups that were not used as model inputs (sex, race/ethnicity), and across age bands (which is a model input, but where consistency across the age range is still a relevant check given its outsized influence on the outcome).

**Rationale:** a model with strong aggregate performance can still perform unevenly across subgroups. A subgroup-level audit is necessary to characterize this directly rather than assume uniform performance from an overall metric.

Sex and race/ethnicity are not included as predictors; they are used here only to stratify the evaluation.

In [ ]:
SEX_LABELS = {1: "Male", 2: "Female"}
RACE_LABELS = {
    1: "Mexican American", 2: "Other Hispanic", 3: "Non-Hispanic White",
    4: "Non-Hispanic Black", 6: "Non-Hispanic Asian", 7: "Other/Multiracial",
}

def fairness_breakdown(pipeline, X_test, y_test, group_series, group_labels):
    """Compute AUROC separately within each subgroup."""
    risk_scores = (pipeline.predict_proba(X_test)[:, 1]
                   if hasattr(pipeline, "predict_proba")
                   else pipeline.decision_function(X_test))
    y_arr = np.asarray(y_test)
    group_arr = np.asarray(group_series)

    rows = {}
    for code, label in group_labels.items():
        mask = group_arr == code
        if mask.sum() == 0 or len(np.unique(y_arr[mask])) < 2:
            continue
        rows[label] = {"n": mask.sum(), "AUROC": roc_auc_score(y_arr[mask], risk_scores[mask])}
    return pd.DataFrame(rows).T

sex_test = df.loc[X_test.index, "RIAGENDR"]
race_test = df.loc[X_test.index, "RIDRETH3"]

sex_fairness = fairness_breakdown(best_pipeline, X_test, y_test, sex_test, SEX_LABELS)
race_fairness = fairness_breakdown(best_pipeline, X_test, y_test, race_test, RACE_LABELS)

print("--- Model discrimination by sex ---")
print(sex_fairness)
print("\n--- Model discrimination by race/ethnicity ---")
print(race_fairness)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].barh(sex_fairness.index, sex_fairness["AUROC"], color="steelblue")
axes[0].set_xlim(0.5, 1.0)
axes[0].set_xlabel("AUROC")
axes[0].set_title("Discrimination by Sex")
axes[0].axvline(0.5, color="gray", linestyle="--", linewidth=1)

axes[1].barh(race_fairness.index, race_fairness["AUROC"], color="darkorange")
axes[1].set_xlim(0.5, 1.0)
axes[1].set_xlabel("AUROC")
axes[1].set_title("Discrimination by Race/Ethnicity")
axes[1].axvline(0.5, color="gray", linestyle="--", linewidth=1)

plt.tight_layout()
plt.savefig("figures/fairness_check.png", dpi=150)
plt.show()

## 14. Global Feature Importance (SHAP)

SHapley Additive exPlanations (SHAP) quantify each predictor's contribution to individual predictions, aggregated here to characterize overall feature importance and directionality. This analysis is restricted to the best-performing tree-based model (Random Forest, XGBoost, or LightGBM), since SHAP's `TreeExplainer` provides fast, exact attribution for tree ensembles, whereas non-tree models require a slower approximation method.

No predictor is manually added to or removed from this analysis to match any external reference; the set interpreted here is exactly the set selected by RFE in Section 7.

In [ ]:
tree_methods = ["Random Forest", "XGBoost", "LightGBM"]
best_tree_model_name = performance_table.loc["AUROC", tree_methods].idxmax()
tree_pipeline = trained_models[best_tree_model_name]["pipeline"]
tree_model_only = tree_pipeline.named_steps["predict"]

print(f"Interpreting the best tree-based model: {best_tree_model_name}")

X_shap_sample = X_test.sample(n=min(500, len(X_test)), random_state=RANDOM_SEED)
explainer = shap.TreeExplainer(tree_model_only)
raw_shap_values = explainer.shap_values(X_shap_sample)

if isinstance(raw_shap_values, list):
    shap_values = raw_shap_values[1]
elif isinstance(raw_shap_values, np.ndarray) and raw_shap_values.ndim == 3:
    shap_values = raw_shap_values[:, :, 1]
else:
    shap_values = raw_shap_values

X_shap_readable = X_shap_sample.copy()
X_shap_readable.columns = to_readable(X_shap_readable.columns)

plt.figure()
shap.summary_plot(shap_values, X_shap_readable, show=False)
plt.title(f"SHAP Summary -- What Drives {best_tree_model_name}'s Predictions?")
plt.tight_layout()
plt.savefig("figures/shap_summary.png", dpi=150, bbox_inches="tight")
plt.show()

top_factors = (pd.DataFrame(np.abs(shap_values), columns=to_readable(X_shap_sample.columns))
               .mean().sort_values(ascending=False).head(8))
print("\nTop predictors by mean absolute SHAP value:")
print(top_factors)

## 15. Local Explanation (LIME)

Local Interpretable Model-agnostic Explanations (LIME) provide an explanation for a single prediction, complementing the global view from SHAP. This is useful for auditing whether individual predictions are consistent with domain expectations.

In [ ]:
lime_explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_train.values,
    feature_names=to_readable(X_train.columns),
    class_names=["No CVD", "Has CVD"],
    mode="classification",
    random_state=RANDOM_SEED,
)

example_index = 0
instance = X_test.iloc[example_index].values
true_status = "Has CVD" if y_test.iloc[example_index] == 1 else "No CVD"

explanation = lime_explainer.explain_instance(instance, tree_model_only.predict_proba, num_features=8)
fig = explanation.as_pyplot_figure()
fig.suptitle(f"Example person #{example_index + 1} (actual status: {true_status})", y=1.05)
fig.savefig("figures/lime_example.png", dpi=150, bbox_inches="tight")
plt.show()

## 16. Model Artifact Export

The best-performing model (using calibrated probabilities, per Section 12) is serialized for use by the accompanying Streamlit application.

In [ ]:
os.makedirs("app/model", exist_ok=True)
artifact = {
    "pipeline": best_pipeline,
    "calibrated_pipeline": calibrated_model,
    "model_name": best_model_name,
    "selected_features": selected_features,
}
with open("app/model/best_model_artifact.pkl", "wb") as f:
    pickle.dump(artifact, f)

print("Model artifact saved to app/model/best_model_artifact.pkl")
print("\nPipeline execution complete. All figures saved to the figures/ folder.")

## Summary of Findings

- All five models achieved comparable discrimination, with overlapping bootstrap confidence intervals, indicating that model choice is not a material factor for this task and dataset.
- Age, total cholesterol, and waist circumference were the strongest predictors overall. Notably, all three newly added clinical predictors -- estimated kidney function (eGFR), diabetes status, and HDL cholesterol -- ranked among the top 6 predictors by SHAP importance, each showing the clinically expected direction of effect (lower eGFR, presence of diabetes, and lower HDL each increase predicted risk).
- Smoking status, also newly added, was not selected by the feature selection step in this run. This does not necessarily indicate smoking is unimportant for CVD risk in general -- rather, its signal likely overlaps substantially with the other newly added predictors (diabetes, kidney function, HDL), and Recursive Feature Elimination tends to retain one representative of a correlated cluster rather than all of them. This is discussed further in `model_card.md`.
- Raw predicted probabilities substantially overestimated risk as a direct consequence of class-balancing during training; this was corrected via post-hoc calibration.
- The subgroup audit identified measurable differences in model discrimination across sex and race/ethnicity groups. These findings are reported directly, with sample-size caveats, in `model_card.md`.
- All associations reported here are observational. The cross-sectional design and self-reported outcome preclude causal interpretation.